# 00 — Pilot: full pipeline end-to-end at tiny scale

**Runner: Tomer only. Do not distribute until this passes.**

Purpose: exercise every stage of the real pipeline on a small sample, and answer the one
open research risk before three other people spend ~14 GPU-hours.

**The critical question this notebook answers:** confidence on LMEnt-170M-6E goes
`0.167 (step0) -> 0.471 (step10000)` and is then flat for 648k steps. All of the onset happens
in the first 10k steps. `LMEnt-1B-6E` is the only released model with checkpoints in that window
(step1000-step9000). So: **is confidence already saturated at step1000?**
- If it rises gradually over step1000-10000, hypothesis H2 (onset timing vs frequency) is measurable.
- If it is already ~0.47 at step1000, onset is faster than public checkpoints can resolve, and we
  pivot the paper to gap *magnitude*. Either way we need to know now.

**Setup:** Kaggle, GPU P100, **Internet must be ON** (Settings -> Internet).
**Runtime:** ~25 minutes. **Do not use `/kaggle/working` for checkpoints** (20 GB output cap).

At the end, this prints a `PILOT REPORT`. Copy that whole block back to Claude.

## 1. Environment

In [ ]:
import sys, os, subprocess, json, time, gc, shutil, platform

# ---- Paths FIRST: HF_HOME must be set before huggingface_hub is imported, or it is ignored.
ON_KAGGLE = os.path.isdir("/kaggle")

def _writable(d):
    """Return d if we can actually create and write inside it, else None."""
    try:
        os.makedirs(d, exist_ok=True)
        t = os.path.join(d, ".wtest")
        with open(t, "w") as f: f.write("x")
        os.remove(t)
        return d
    except Exception:
        return None

# Checkpoints go to scratch that does NOT count against Kaggle's 20 GB output cap.
# /kaggle/working is for results only - it is versioned and saved as notebook output.
SCRATCH = None
for cand in (["/kaggle/temp", "/tmp", "/kaggle/working/_scratch"] if ON_KAGGLE
             else ["./_scratch"]):
    SCRATCH = _writable(cand)
    if SCRATCH: break
assert SCRATCH, "no writable scratch directory found"

OUT = _writable("/kaggle/working" if ON_KAGGLE else "./_out")
assert OUT, "no writable output directory found"

TMP = os.path.join(SCRATCH, "ckpt"); os.makedirs(TMP, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(SCRATCH, "hf")     # before any HF import
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

if SCRATCH.startswith("/kaggle/working"):
    print("WARNING: using /kaggle/working for checkpoints - watch the 20 GB output cap.")
    print("         Each checkpoint is deleted after use, so it should still fit.")

def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# olmo2 architecture requires transformers >= 4.47
try:
    import transformers
    from packaging.version import parse as V
    if V(transformers.__version__) < V("4.47"):
        print(f"transformers {transformers.__version__} too old -> upgrading")
        pipq("-U", "transformers>=4.47"); os.execv(sys.executable, [sys.executable] + sys.argv)
except ImportError:
    pipq("-U", "transformers>=4.47")

import torch, transformers, numpy as np, pandas as pd
try:
    import pyarrow
except ImportError:
    pipq("pyarrow"); import pyarrow

ENV = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory/1e9, 1) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV, indent=2))
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2"

def preflight_gpu():
    """Recent PyTorch dropped sm_50/60/70 kernels. Fail in seconds, not mid-sweep."""
    name  = torch.cuda.get_device_name(0)
    cap   = torch.cuda.get_device_capability(0)
    sm    = "sm_" + str(cap[0]) + str(cap[1])
    archs = list(torch.cuda.get_arch_list())
    print("GPU:", name, "| capability:", sm)
    print("this torch was built for:", ", ".join(archs))
    if sm not in archs:
        print("")
        print("=" * 72)
        print("STOP - this PyTorch build has no kernels for this GPU (" + name + ", " + sm + ").")
        print("Recent PyTorch dropped Maxwell/Pascal/Volta (sm_50 / sm_60 / sm_70).")
        print("Kaggle's P100 is sm_60 and is currently BROKEN - Kaggle/docker-python issue 1546.")
        print("")
        print("FIX: Settings -> Accelerator -> 'GPU T4 x2', then Run All again.")
        print("T4 is sm_75, supported, and FASTER than P100 here (fp16 tensor cores).")
        print("=" * 72)
        raise SystemExit("incompatible GPU (" + sm + ") - switch the accelerator to T4 x2")
    try:                                    # arch_list can still miss things
        emb = torch.nn.Embedding(16, 8).cuda()
        emb(torch.zeros(2, dtype=torch.long, device="cuda")).float().sum().item()
        a = torch.randn(64, 64, device="cuda", dtype=torch.float16)
        (a @ a).float().sum().item()
    except Exception as ex:
        print("")
        print("=" * 72)
        print("STOP - " + sm + " is in this torch build, but a real GPU op still failed:")
        print("  " + type(ex).__name__ + ": " + str(ex))
        print("This is NOT the usual architecture mismatch. Restart the session first.")
        print("If it persists, try Settings -> Accelerator -> 'GPU T4 x2'.")
        print("=" * 72)
        raise SystemExit("GPU smoke test failed: " + type(ex).__name__)
    print("GPU preflight: OK (fp16 matmul + embedding lookup)")
    return sm, archs

SM, ARCHS = preflight_gpu()
ENV["gpu_capability"] = SM
ENV["torch_arch_list"] = ARCHS

import shutil as _sh
free_gb = _sh.disk_usage(SCRATCH).free / 1e9
print("")
print(f"on_kaggle: {ON_KAGGLE}")
print(f"checkpoints -> {TMP}  ({free_gb:.0f} GB free)")
print(f"results     -> {OUT}")
assert free_gb > 12, (f"Only {free_gb:.0f} GB free in {SCRATCH}. Need ~12 GB headroom for a "
                      f"5.3 GB LMEnt-1B checkpoint plus its download temp files.")
ENV["scratch"] = TMP; ENV["out"] = OUT; ENV["free_gb"] = round(free_gb, 1)

In [ ]:
# Internet check - the most common Kaggle failure
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
    print("Internet: OK")
except Exception as e:
    raise SystemExit(f"Internet is OFF. Kaggle -> Settings -> Internet -> On (needs phone verification).\n{e}")

## 2. Config

Pilot scale. The real runs use `N_PER_BUCKET=200` and the full checkpoint list.

In [ ]:
SEED           = 0
N_PER_BUCKET   = 24          # pilot: 24 x 10 buckets = 240 facts (real run: 200)
N_CAND         = 10          # gold + 9 distractors -> chance = 0.100
GEN_TOKENS     = 10
BATCH          = 64

# THE CRITICAL TEST: dense early window on the only model that has one.
PILOT_MODEL = "dhgottesman/LMEnt-1B-6E"
PILOT_STEPS = ["step0", "step1000", "step3000", "step6000", "step10000"]

BUCKET_EDGES  = [-1, 0, 1, 2, 4, 7, 13, 25, 60, 200, 10**9]
BUCKET_LABELS = ["0","1","2","3-4","5-7","8-13","14-25","26-60","61-200","200+"]

TEMPLATES = {
    "director":       "The director of {s} is",
    "screenwriter":   "The screenwriter of {s} is",
    "genre":          "The genre of {s} is",
    "producer":       "The producer of {s} is",
    "author":         "The author of {s} is",
    "composer":       "The composer of {s} is",
    "country":        "{s} is located in the country of",
    "capital":        "The capital of {s} is",
    "place of birth": "{s} was born in the city of",
    "father":         "The father of {s} is",
    "sport":          "{s} plays the sport of",
    "occupation":     "The occupation of {s} is",
    "capital of":     "{s} is the capital of",
    "religion":       "The religion of {s} is",
    "mother":         "The mother of {s} is",
    "color":          "The color of {s} is",
}
REPORT = {"env": ENV, "config": {"n_per_bucket": N_PER_BUCKET, "n_cand": N_CAND,
                                 "model": PILOT_MODEL, "steps": PILOT_STEPS}}
print(f"{N_PER_BUCKET * len(BUCKET_LABELS)} facts x {len(PILOT_STEPS)} checkpoints")

## 3. Load fact metadata

Column-selective remote parquet read. The `*_chunks` list columns are the bulk of the 2.5 GB
dataset and we skip them entirely, so this pulls only a few MB.

In [ ]:
from huggingface_hub import HfFileSystem
import pyarrow.parquet as pq

COLS = ["id","subj","prop","obj","subj_id","obj_id","s_pop","o_pop",
        "question","possible_answers",
        "subject_num_chunks","answer_num_chunks","num_shared_chunks"]

t0 = time.time()
fs = HfFileSystem()
parts = []
for i in range(9):
    p = f"datasets/dhgottesman/popqa-kas/data/train-0000{i}-of-00009.parquet"
    with fs.open(p, "rb") as fh:
        parts.append(pq.ParquetFile(fh).read(columns=COLS).to_pandas())
    print(f"  shard {i}: {len(parts[-1])} rows", flush=True)
facts = pd.concat(parts, ignore_index=True)
print(f"\n{len(facts)} facts in {time.time()-t0:.0f}s")

assert len(facts) == 14267, f"expected 14267 facts, got {len(facts)}"
assert facts.prop.nunique() == 16, f"expected 16 relations, got {facts.prop.nunique()}"
assert set(facts.prop) <= set(TEMPLATES), f"missing templates: {set(facts.prop)-set(TEMPLATES)}"
print("Relations:", facts.prop.nunique(), "| all have templates")
REPORT["n_facts_total"] = len(facts)

In [ ]:
facts["bucket"] = pd.cut(facts.num_shared_chunks, BUCKET_EDGES, labels=BUCKET_LABELS)
print(facts.bucket.value_counts().reindex(BUCKET_LABELS).to_string())
print("\nControl groups:")
print(f"  subject absent from corpus : {(facts.subject_num_chunks==0).sum()}")
print(f"  subject & object never co-occur: {(facts.num_shared_chunks==0).sum()}")
REPORT["bucket_counts"] = facts.bucket.value_counts().reindex(BUCKET_LABELS).to_dict()

## 4. Build the probe set

Candidate sets are built **once with a fixed seed** and reused across every checkpoint and every
runner. Distractors are drawn from the same relation's object pool (type-consistent),
**token-length-matched** to the gold (limits surface-form bias), and screened against the gold's
alias list so no distractor is secretly correct.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(PILOT_MODEL, subfolder="step10000", use_fast=True)
print("tokenizer ok | pad:", tok.pad_token_id, "| bos:", tok.bos_token_id)

def ntok(s):
    return len(tok(" " + s, add_special_tokens=False)["input_ids"])

def as_list(x):
    """possible_answers arrives as a numpy array; `x or []` would raise on it."""
    if x is None: return []
    return [str(v) for v in list(x)]

rng = np.random.default_rng(SEED)
sample = pd.concat([g.sample(min(N_PER_BUCKET, len(g)), random_state=SEED)
                    for _, g in facts.groupby("bucket", observed=True)]).reset_index(drop=True)

# object pool + cached token lengths per relation
pools = {p: sorted(set(g.obj)) for p, g in facts.groupby("prop")}
pool_len = {p: np.array([ntok(o) for o in pools[p]]) for p in pools}

probes = []
for r in sample.itertuples():
    pool, plens = pools[r.prop], pool_len[r.prop]
    gold_len = ntok(r.obj)
    banned = {r.obj.lower()} | {a.lower() for a in as_list(r.possible_answers)}
    # prefer length-matched distractors, widening tolerance until we have enough
    cands = [r.obj]
    for tol in (0, 1, 2, 99):
        idx = np.flatnonzero(np.abs(plens - gold_len) <= tol)
        rng.shuffle(idx)
        for j in idx:
            c = pool[j]
            if c.lower() in banned or c in cands:
                continue
            cands.append(c)
            if len(cands) == N_CAND:
                break
        if len(cands) == N_CAND:
            break
    if len(cands) < N_CAND:
        continue
    probes.append({
        "fact_id": int(r.id), "subj": r.subj, "prop": r.prop, "obj": r.obj,
        "bucket": str(r.bucket), "n_shared": int(r.num_shared_chunks),
        "n_subject": int(r.subject_num_chunks),
        "prompt": TEMPLATES[r.prop].format(s=r.subj),
        "candidates": cands,                     # index 0 is ALWAYS the gold
        "possible_answers": as_list(r.possible_answers),
    })

print(f"{len(probes)} probes built")
gl = np.array([ntok(p["candidates"][0]) for p in probes])
dl = np.array([ntok(c) for p in probes for c in p["candidates"][1:]])
print(f"gold token len {gl.mean():.2f} vs distractor {dl.mean():.2f}  (should be close)")
assert abs(gl.mean() - dl.mean()) < 0.5, "distractor length matching failed -> length bias"
print("\nExample probe:"); print(json.dumps(probes[0], indent=2)[:600])
REPORT["n_probes"] = len(probes)
REPORT["len_match"] = {"gold": round(float(gl.mean()),3), "distractor": round(float(dl.mean()),3)}

## 5. Scoring

One forward pass per (prompt, candidate). Three scores from the same pass:
- **`norm`** — length-normalized log-prob. **Primary** (only one that gives a random-init model
  chance-level confidence)
- `sum` — raw sum log-prob (reports 0.67 confidence at step0; kept as a documented failure mode)
- `pmi` — `log p(c|prompt) - log p(c|"The answer is")`, robustness check

In [ ]:
import torch.nn.functional as F

def fetch_ckpt(model_id, sub):
    """Download one checkpoint to ephemeral scratch. Returns its directory."""
    from huggingface_hub import snapshot_download
    d = os.path.join(TMP, sub)
    snapshot_download(model_id, allow_patterns=[f"{sub}/*"], local_dir=d)
    return d

def load_from(d, sub, dt):
    from transformers import AutoModelForCausalLM
    try:
        m = AutoModelForCausalLM.from_pretrained(f"{d}/{sub}", dtype=dt)
    except TypeError:                       # transformers < 4.56 uses torch_dtype
        m = AutoModelForCausalLM.from_pretrained(f"{d}/{sub}", torch_dtype=dt)
    return m.to("cuda").eval()

def free_model(m):
    del m; gc.collect(); torch.cuda.empty_cache()

def drop_ckpt(d):
    shutil.rmtree(d, ignore_errors=True)

def all_finite(scored):
    v = np.array([x[0] for x in scored], dtype=float)
    return bool(np.isfinite(v).all()), int((~np.isfinite(v)).sum()), len(v)

@torch.no_grad()
def score_pairs(model, pairs, batch=BATCH):
    """pairs: [(prompt, answer)]. Returns (sum_logprob, n_answer_tokens) per pair."""
    out = []
    for i in range(0, len(pairs), batch):
        chunk = pairs[i:i+batch]
        seqs, alen = [], []
        for p, a in chunk:
            pid = tok(p, add_special_tokens=False)["input_ids"]
            aid = tok(" " + a, add_special_tokens=False)["input_ids"]
            seqs.append([tok.bos_token_id] + pid + aid); alen.append(len(aid))
        L = max(len(s) for s in seqs)
        inp = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(seqs), L), dtype=torch.long)
        for j, s in enumerate(seqs):
            inp[j,:len(s)] = torch.tensor(s); att[j,:len(s)] = 1
        lg = model(input_ids=inp.cuda(), attention_mask=att.cuda()).logits.float()
        lp = F.log_softmax(lg, -1)
        for j, s in enumerate(seqs):
            n, e = alen[j], len(s)
            tgt = torch.tensor(s[e-n:e], device=lp.device)
            tl  = lp[j, e-n-1:e-1, :].gather(-1, tgt[:,None]).squeeze(-1)
            out.append((tl.sum().item(), n))
    return out

@torch.no_grad()
def generate(model, prompts, batch=32):
    """Greedy decode + mean token log-prob of the emitted span."""
    texts, confs = [], []
    tok.padding_side = "left"
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    for i in range(0, len(prompts), batch):
        enc = tok(prompts[i:i+batch], return_tensors="pt", padding=True,
                  add_special_tokens=True).to("cuda")
        o = model.generate(**enc, max_new_tokens=GEN_TOKENS, do_sample=False,
                           output_scores=True, return_dict_in_generate=True,
                           pad_token_id=tok.pad_token_id)
        new = o.sequences[:, enc["input_ids"].shape[1]:]
        st  = torch.stack(o.scores, 1).float()                 # (B, T, V)
        lp  = F.log_softmax(st, -1).gather(-1, new[:,:,None]).squeeze(-1)
        live = (new != tok.pad_token_id).float()
        mlp = (lp*live).sum(1) / live.sum(1).clamp(min=1)
        texts += tok.batch_decode(new, skip_special_tokens=True)
        confs += torch.exp(mlp).tolist()
    tok.padding_side = "right"
    return texts, confs

print("scoring functions defined")

In [ ]:
import re
def norm_txt(s):
    return re.sub(r"[^a-z0-9 ]", "", (s or "").lower()).strip()

PROMPTS   = [p["prompt"] for p in probes]
PAIRS     = [(p["prompt"], c) for p in probes for c in p["candidates"]]
UNIQ_CAND = sorted({c for p in probes for c in p["candidates"]})
NEUTRAL   = [("The answer is", c) for c in UNIQ_CAND]     # deduped for PMI
CIDX      = {c: i for i, c in enumerate(UNIQ_CAND)}
print(f"{len(PAIRS)} scored pairs + {len(NEUTRAL)} neutral + {len(PROMPTS)} generations per checkpoint")

# marginal-prior candidate per probe (corpus-frequency definition), for the mechanism analysis
obj_freq = facts.groupby("obj").answer_num_chunks.max().to_dict()
for p in probes:
    p["prior_idx"] = int(np.argmax([obj_freq.get(c, 0) for c in p["candidates"]]))
print("prior_idx == gold for", sum(p["prior_idx"]==0 for p in probes), "/", len(probes), "probes")

## 5b. Choose a numeric dtype — empirically, on a TRAINED checkpoint

**This step exists because of a real failure.** LMEnt checkpoints are stored in fp32. The 1B
model's peak activation is ~3.06e6 (at `layers.12.mlp.down_proj`), which is 47x above fp16's
65504 ceiling, so **fp16 turns every logit into NaN**. A randomly initialised `step0` does *not*
overflow, so testing on step0 alone hides the bug completely.

So we calibrate on the first **trained** checkpoint, verify outputs are finite, and time each
option. T4 (sm_75) has no native bf16 hardware, so bf16 may be emulated and slower than fp32 —
we measure rather than assume.

In [ ]:
CAL_STEP = next(s for s in PILOT_STEPS if int(s[4:]) > 0)   # never step0
print("dtype calibration on", CAL_STEP, "(a trained checkpoint)")
cal_dir = fetch_ckpt(PILOT_MODEL, CAL_STEP)

CANDIDATES = [("float16", torch.float16),      # expected to fail; kept to document it
              ("bfloat16", torch.bfloat16),
              ("float32", torch.float32)]
probe_pairs = PAIRS[:128]
dtype_report = {}

for nm, dt in CANDIDATES:
    try:
        m = load_from(cal_dir, CAL_STEP, dt)
        torch.cuda.synchronize(); t0 = time.time()
        out = score_pairs(m, probe_pairs)
        torch.cuda.synchronize(); sec = time.time() - t0
        ok, nbad, ntot = all_finite(out)
        dtype_report[nm] = {"finite": ok, "sec": round(sec, 2), "nonfinite": f"{nbad}/{ntot}"}
        free_model(m)
    except Exception as ex:
        dtype_report[nm] = {"finite": False, "sec": None, "error": type(ex).__name__ + ": " + str(ex)[:160]}
    print(" ", nm, dtype_report[nm], flush=True)

usable = [(nm, dtype_report[nm]["sec"]) for nm, _ in CANDIDATES if dtype_report[nm]["finite"]]
assert usable, "No dtype produced finite outputs - stop and report this."
DTYPE_NAME = min(usable, key=lambda x: x[1])[0]
DTYPE = dict(CANDIDATES)[DTYPE_NAME]
print("")
print("chosen dtype:", DTYPE_NAME, "(fastest option with finite outputs)")
REPORT["dtype_calibration"] = dtype_report
REPORT["dtype_chosen"] = DTYPE_NAME

## 6. Run the checkpoint sweep

Download -> score -> save -> delete, one checkpoint at a time. Resumes if the session dies.

In [ ]:
TIMING = {}

def run_checkpoint(model_id, sub):
    f = os.path.join(OUT, f"pilot__{model_id.split('/')[-1]}__{sub}.parquet")
    if os.path.exists(f):
        print(f"{sub}: cached"); return pd.read_parquet(f)
    t0 = time.time()
    d  = cal_dir if sub == CAL_STEP else fetch_ckpt(model_id, sub)
    t_dl = time.time() - t0
    model = load_from(d, sub, DTYPE)

    t1 = time.time()
    sc  = score_pairs(model, PAIRS)
    nsc = score_pairs(model, NEUTRAL)
    gtxt, gconf = generate(model, PROMPTS)
    t_gpu = time.time() - t1
    free_model(model)
    if sub != CAL_STEP:
        drop_ckpt(d)

    # HARD GUARD: never let non-finite scores reach the analysis.
    for label, arr in [("probe", sc), ("neutral", nsc)]:
        ok, nbad, ntot = all_finite(arr)
        if not ok:
            raise RuntimeError(f"{sub}: {nbad}/{ntot} non-finite {label} scores "
                               f"under dtype={DTYPE_NAME}. Numerical overflow - do not trust "
                               f"any downstream number.")

    neutral_lp = {c: nsc[CIDX[c]][0] for c in UNIQ_CAND}
    rows = []
    for qi, p in enumerate(probes):
        blk  = sc[qi*N_CAND:(qi+1)*N_CAND]
        s    = np.array([b[0] for b in blk]); nt = np.array([b[1] for b in blk])
        nlp  = np.array([neutral_lp[c] for c in p["candidates"]])
        r = {"fact_id": p["fact_id"], "model": model_id.split("/")[-1], "step": int(sub[4:]),
             "bucket": p["bucket"], "n_shared": p["n_shared"], "n_subject": p["n_subject"],
             "prior_idx": p["prior_idx"]}
        for name, sco in [("norm", s/nt), ("sum", s), ("pmi", s-nlp)]:
            pr  = np.exp(sco - sco.max()); pr /= pr.sum()
            top = int(np.argmax(pr))
            r[f"acc_{name}"]   = int(top == 0)
            r[f"conf_{name}"]  = float(pr.max())
            r[f"pgold_{name}"] = float(pr[0])
            r[f"top_{name}"]   = top
        r["chose_prior"] = int(r["top_norm"] == p["prior_idx"] and r["top_norm"] != 0)
        g = gtxt[qi]
        r["gen_text"]    = g
        r["gen_correct"] = int(any(norm_txt(a) and norm_txt(a) in norm_txt(g)
                                   for a in p["possible_answers"]))
        r["gen_conf"]    = float(gconf[qi])
        rows.append(r)
    df = pd.DataFrame(rows)

    numeric = [c for c in df.columns if df[c].dtype.kind == "f"]
    nbad = int(df[numeric].isna().sum().sum())
    if nbad:
        raise RuntimeError(f"{sub}: {nbad} NaN values in the results table - refusing to save.")

    df.to_parquet(f, index=False)
    TIMING[sub] = {"download_s": round(t_dl, 1), "gpu_s": round(t_gpu, 1),
                   "total_s": round(time.time() - t0, 1)}
    print(f"{sub}: {TIMING[sub]['total_s']:6.0f}s "
          f"(dl {TIMING[sub]['download_s']:.0f}s, gpu {TIMING[sub]['gpu_s']:.0f}s) | "
          f"acc={df.acc_norm.mean():.3f} conf={df.conf_norm.mean():.3f}", flush=True)
    return df

res = pd.concat([run_checkpoint(PILOT_MODEL, s) for s in PILOT_STEPS], ignore_index=True)
drop_ckpt(cal_dir)
res.to_parquet(os.path.join(OUT, "pilot_all.parquet"), index=False)
REPORT["timing"] = TIMING

# Global guard: every checkpoint must have produced usable rows.
missing = set(int(s[4:]) for s in PILOT_STEPS) - set(res.step.unique())
assert not missing, f"missing checkpoints in results: {sorted(missing)}"
numeric = [c for c in res.columns if res[c].dtype.kind == "f"]
assert not res[numeric].isna().any().any(), "NaN in the pooled results - do not interpret."
print(f"\n{len(res)} rows, {res.step.nunique()} checkpoints, all finite")

## 7. Sanity checks

These must pass. A failure here means a bug, not a finding.

In [ ]:
checks = {}
s0 = res[res.step == 0]

# 1. random init must be at chance accuracy
checks["step0_acc_is_chance"] = bool(abs(s0.acc_norm.mean() - 1/N_CAND) < 0.06)
# 2. random init must not be confident under the primary metric
checks["step0_conf_is_low"]   = bool(s0.conf_norm.mean() < 0.30)
# 3. the primary metric must be the best-calibrated of the three at random init
checks["norm_best_calibrated_at_init"] = bool(
    s0.conf_norm.mean() <= s0.conf_sum.mean() + 1e-6 and
    s0.conf_norm.mean() <= s0.conf_pmi.mean() + 1e-6)
# 4. accuracy must rise with exposure at the final pilot checkpoint
last = res[res.step == res.step.max()]
lo = last[last.bucket.isin(["0","1","2"])].acc_norm.mean()
hi = last[last.bucket.isin(["61-200","200+"])].acc_norm.mean()
checks["acc_rises_with_exposure"] = bool(hi > lo + 0.05)
# 5. generation must produce non-empty AND VARIED text.
#    NaN logits make argmax collapse to token 0, giving identical non-empty strings for
#    every fact - which the old "non-empty" check happily passed. Hence the variety test.
checks["generation_nonempty"] = bool((last.gen_text.str.strip().str.len() > 0).mean() > 0.9)
checks["generation_varies"]   = bool(last.gen_text.nunique() > 0.3 * len(last))
# 6. nothing anywhere may be non-finite
_num = [c for c in res.columns if res[c].dtype.kind == "f"]
checks["all_values_finite"]      = bool(not res[_num].isna().any().any())
checks["all_checkpoints_present"] = bool(res.step.nunique() == len(PILOT_STEPS))

print(f"step0 (random init) — correct confidence here is chance = {1/N_CAND:.3f}")
for m in ["norm", "sum", "pmi"]:
    print(f"  {m:>4}: acc={s0[f'acc_{m}'].mean():.3f}  conf={s0[f'conf_{m}'].mean():.3f}")
REPORT["step0_scorers"] = {m: {"acc": round(float(s0[f'acc_{m}'].mean()),3),
                               "conf": round(float(s0[f'conf_{m}'].mean()),3)}
                           for m in ["norm","sum","pmi"]}
print(f"final: acc rare={lo:.3f} vs frequent={hi:.3f}\n")
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
REPORT["sanity"] = checks
if not all(checks.values()):
    print("\n>>> Some checks FAILED. Send the report to Claude before running anything else. <<<")

## 8. THE CRITICAL TEST — is onset resolvable?

In [ ]:
curve = res.groupby("step")[["acc_norm","conf_norm"]].mean().round(3)
print("Aggregate over training steps:\n"); print(curve.to_string())

if curve.isna().any().any() or not {0, 1000}.issubset(set(curve.index)):
    print("")
    print("NO VERDICT: the curve has missing or non-finite values, or step0/step1000 are absent.")
    print("A verdict from this data would be meaningless. Send the report to Claude as-is.")
    REPORT["onset"] = {"verdict": "UNDETERMINED - non-finite or incomplete curve",
                       "curve": curve.to_dict()}
    raise SystemExit("cannot compute a verdict from incomplete data")

c0    = curve.loc[0, "conf_norm"]
c1000 = curve.loc[1000, "conf_norm"]
cmax  = curve["conf_norm"].max()
assert cmax > c0, f"confidence never rises above its step0 value ({c0:.3f}) - suspicious"
frac  = (c1000 - c0) / (cmax - c0)

print(f"\nconfidence at step0    : {c0:.3f}")
print(f"confidence at step1000 : {c1000:.3f}")
print(f"confidence at plateau  : {cmax:.3f}")
print(f"fraction of the rise already complete by step1000: {frac:.1%}\n")

if frac > 0.85:
    verdict = ("SATURATED — onset is faster than public checkpoints can resolve. "
               "H2 (onset timing) closes as a negative result; pivot the paper to gap MAGNITUDE.")
elif frac < 0.5:
    verdict = ("RESOLVABLE — onset unfolds across step1000-10000. "
               "H2 is measurable; the dense early window is the paper's centrepiece.")
else:
    verdict = ("PARTIAL — most of the rise is early but not instant. "
               "Usable, but add step2000/4000/5000/7000/8000 to the dense sweep.")
print("VERDICT:", verdict)
REPORT["onset"] = {"conf_step0": float(c0), "conf_step1000": float(c1000),
                   "conf_plateau": float(cmax), "frac_by_1000": float(frac),
                   "verdict": verdict}

## 9. Per-bucket picture and the mechanism probe

In [ ]:
pd.set_option("display.width", 250)
for m in ["acc_norm", "conf_norm"]:
    p = res.groupby(["bucket","step"], observed=True)[m].mean().unstack("step").reindex(BUCKET_LABELS).round(3)
    print(f"\n=== {m} ===\n{p.to_string()}")

gap = (res.groupby(["bucket","step"], observed=True)["conf_norm"].mean()
       - res.groupby(["bucket","step"], observed=True)["acc_norm"].mean()).unstack("step").reindex(BUCKET_LABELS).round(3)
print(f"\n=== GAP (conf - acc) ===\n{gap.to_string()}")
REPORT["gap_final"] = gap[gap.columns[-1]].to_dict()

# mechanism: do confidently-wrong predictions collapse onto the relation's marginal prior?
w = res[(res.acc_norm == 0) & (res.step > 0)]
print(f"\nPrior-collapse rate among wrong answers: {w.chose_prior.mean():.3f} "
      f"(chance ~ {1/(N_CAND-1):.3f})")
print(w.groupby("bucket", observed=True).chose_prior.mean().reindex(BUCKET_LABELS).round(3).to_string())
REPORT["prior_collapse_overall"] = float(w.chose_prior.mean())

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for b in BUCKET_LABELS:
    d = res[res.bucket == b].groupby("step")[["acc_norm","conf_norm"]].mean()
    if d.empty: continue
    ax[0].plot(d.index, d.acc_norm,  marker="o", ms=3, label=b)
    ax[1].plot(d.index, d.conf_norm, marker="o", ms=3, label=b)
    ax[2].plot(d.index, d.conf_norm - d.acc_norm, marker="o", ms=3, label=b)
for a, t in zip(ax, ["Accuracy", "Confidence", "Gap (conf - acc)"]):
    a.set_title(t); a.set_xlabel("training step"); a.grid(alpha=.3)
ax[0].axhline(1/N_CAND, ls="--", c="k", lw=.8)
ax[2].axhline(0, ls="--", c="k", lw=.8)
ax[2].legend(fontsize=7, ncol=2, title="exposure")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "pilot_curves.png"), dpi=130); plt.show()

## 10. Timing extrapolation and the pilot report

In [ ]:
per_ck = REPORT.setdefault("timing", {})
# scale pilot timing to the real run (200/bucket instead of N_PER_BUCKET/bucket)
scale = (200 * len(BUCKET_LABELS)) / max(len(probes), 1)
print(f"Real run is {scale:.1f}x this pilot's probe volume.")
print("Estimate per checkpoint at full scale: measure from the per-step times printed in Section 6,")
print("multiply the scoring portion by the factor above (download time does not scale).")

print("\nExample generations at the final pilot checkpoint:")
for r in res[res.step == res.step.max()].head(8).itertuples():
    p = next(x for x in probes if x["fact_id"] == r.fact_id)
    print(f"  [{r.bucket:>6}] {p['prompt']!r} -> {r.gen_text.strip()[:45]!r}  "
          f"(gold={p['obj']!r}, ok={r.gen_correct}, conf={r.gen_conf:.2f})")

In [ ]:
REPORT["generated"] = time.strftime("%Y-%m-%d %H:%M:%S")
with open(os.path.join(OUT, "pilot_report.json"), "w") as f:
    json.dump(REPORT, f, indent=2, default=str)

print("=" * 72)
print("PILOT REPORT  —  copy everything below back to Claude")
print("=" * 72)
print(json.dumps(REPORT, indent=2, default=str))
print("=" * 72)
print("\nFiles in", OUT, ":")
for f in sorted(os.listdir(OUT)):
    print(f"  {f}  ({os.path.getsize(os.path.join(OUT,f))/1e6:.2f} MB)")
print("\nAlso send: pilot_curves.png, and the Section 8 VERDICT line.")